<a href="https://colab.research.google.com/github/Ivan8Garcia/FlightOnTime/blob/main/Curso_Agentes_de_IA_y_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Conexion con LLM´s**

In [1]:
!pip install -q langchain langchain-google-genai google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 5.8 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
GEMINI_API_KEY=userdata.get('Gemini_API_key')

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm= ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0,
    google_api_key=GEMINI_API_KEY
)

In [4]:
respuesta= llm.invoke("que es el RAG en IA?")
respuesta

AIMessage(content=[{'type': 'text', 'text': '**RAG** son las siglas en inglés de **Retrieval-Augmented Generation** (en español: **Generación Aumentada por Recuperación**).\n\nEs una técnica de Inteligencia Artificial que permite a un modelo de lenguaje (como GPT-4) consultar fuentes de información externas y actualizadas antes de generar una respuesta.\n\nAquí te explico detalladamente qué es, por qué es importante y cómo funciona:\n\n---\n\n### 1. ¿Por qué es necesario el RAG? (El problema)\nLos modelos de lenguaje tradicionales (LLMs) tienen dos limitaciones principales:\n*   **Fecha de corte:** Solo saben lo que aprendieron durante su entrenamiento. Si les preguntas por algo que pasó ayer, no lo sabrán.\n*   **Alucinaciones:** A veces inventan datos con mucha seguridad si no tienen la información precisa.\n*   **Falta de datos privados:** No conocen los documentos internos de tu empresa o tus archivos personales.\n\n### 2. ¿Cómo funciona el RAG? (La solución)\nImagina que el LLM es

In [5]:
PROMPT_TRIAJE = """
	Eres un especialista en triaje del Service Desk para políticas internas.
	Dado el mensaje del usuario, devuelve SÓLO un JSON con:\n
	{\n
	 "decision": "AUTO_RESOLVER" | "PEDIR_INFO" | "ABRIR_TICKET",\n
	 "urgencia": "BAJA" | "MEDIANA" | "ALTA",\n
	 "campos_faltantes": ["..."]\n
	}\n
	Reglas:\n
	- **AUTO_RESOLVER**: Preguntas claras sobre las reglas o procedimientos descritos en las políticas (Ej.: "¿Puedo reembolsar el internet para mi oficina en casa?", "¿Cómo funciona la política de alimentación mientras viajo?").\n
	- **PEDIR_INFO**: Mensajes imprecisos o sin información para identificar el tema o el contexto (Ej.: "Necesito ayuda con una política", "Tengo una pregunta general").\n
	- **ABRIR_TICKET**: Solicitudes de excepciones, autorización, aprobación o acceso especial, o cuando el usuario solicita explícitamente abrir un ticket (Ej.: "Quiero una excepción para trabajar remotamente durante 5 días", "Solicito autorización para archivos adjuntos externos", "Por favor, abra un ticket con RR. HH.").\n
	Analiza el mensaje y decide la acción más adecuada.
"""



In [6]:
from typing import Literal,List,Dict
from pydantic import BaseModel,Field


class TriajeOut(BaseModel):
  decision: Literal["AUTO_RESOLVER", "PEDIR_INFO","ABRIR_TICKET"]
  urgencia: Literal["BAJA","MEDIANA","ALTA"]
  campos_faltantes:List[str]=Field(default_factory=list)

In [7]:
from langchain_core.messages import SystemMessage,HumanMessage

chain_de_triaje= llm.with_structured_output(TriajeOut)

def triaje(mensaje:str) -> Dict:
  salida: TriajeOut= chain_de_triaje.invoke(
      [
          SystemMessage(content=PROMPT_TRIAJE),
          HumanMessage(content=mensaje)
      ]
  )
  return salida.model_dump()

In [8]:
mensajes_de_prueba=[
    "Puedo obterner un reembolso por el internet de mi home office?",
    "Quiero una excepcion  para teletrabajar durante 5 dias",
    "Como funciona la politica de comidas para viajes?",
    "Existe una politica para anticipos de vacaciones?",
    "Quien fue napoleon bonaparte?"
]

for pregunta in mensajes_de_prueba:
  r=triaje(pregunta)
  print(f"{pregunta} -> {r}")

Puedo obterner un reembolso por el internet de mi home office? -> {'decision': 'AUTO_RESOLVER', 'urgencia': 'BAJA', 'campos_faltantes': []}
Quiero una excepcion  para teletrabajar durante 5 dias -> {'decision': 'ABRIR_TICKET', 'urgencia': 'MEDIANA', 'campos_faltantes': ['Motivo de la solicitud de excepción', 'Fechas específicas para el teletrabajo']}
Como funciona la politica de comidas para viajes? -> {'decision': 'AUTO_RESOLVER', 'urgencia': 'BAJA', 'campos_faltantes': []}
Existe una politica para anticipos de vacaciones? -> {'decision': 'AUTO_RESOLVER', 'urgencia': 'BAJA', 'campos_faltantes': []}
Quien fue napoleon bonaparte? -> {'decision': 'PEDIR_INFO', 'urgencia': 'BAJA', 'campos_faltantes': ['contexto relacionado con políticas internas']}


#**RAG**

In [20]:
!pip install -q langchain_community faiss-cpu langchain-text-splitters pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 23.9 MB/s eta 0:00:00


In [21]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

docs=[]

for documento in Path("/content/").glob("*.pdf"):
    try:
        loader = PyMuPDFLoader(str(documento))
        docs.extend(loader.load())
        print(f"Archivo cargado: {documento.name}")
    except Exception as e:
        print(f"Error cargando archivo: {documento.name}: {e}")

print(f"Total de documentos cargados: {len(docs)}")


Archivo cargado: Política de Reembolsos (Viajes y Gastos).pdf
Archivo cargado: Política de Teletrabajo (Home Office).pdf
Archivo cargado: Política de Uso de Correo Electrónico y Seguridad de la Información.pdf
Total de documentos cargados: 3


In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter= RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=30)
chunks = splitter.split_documents(docs)

In [24]:
for chunk in chunks:
  print(chunk)
  print("-------------------")

page_content='Política de Reembolsos (Viajes y 
Gastos) 
1. Objetivo Establecer las directrices y procedimientos para el reembolso de gastos 
incurridos por los empleados en el ejercicio de sus funciones oficiales, asegurando la 
transparencia, equidad y cumplimiento fiscal.' metadata={'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '/content/Política de Reembolsos (Viajes y Gastos).pdf', 'file_path': '/content/Política de Reembolsos (Viajes y Gastos).pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Política de Reembolsos (Viajes y Gastos)', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}
-------------------
page_content='2. Ámbito de Aplicación Esta política se aplica a todos los empleados fijos y temporales 
que incurran en gastos en nombre de la empresa. 
3. Gastos de Viaje 
●​ Transporte Aéreo: Se reembolsarán vuelos en clase económica. Cualquier mejora' met

In [25]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

modelo_embeddings= GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GEMINI_API_KEY
)

In [26]:
from langchain_community.vectorstores import FAISS

vectorstore= FAISS.from_documents(chunks,modelo_embeddings)

retriever= vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold":0.3, "k":4}
)

In [35]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

prompt_rag = ChatPromptTemplate(
    [
        ("system",
         """
        Eres el especialista en RR.HH. de la empresa Carraro Desarrollo de Software.
        Responde siempre utilizando los conocimientos del contexto que te fue pasado.
        Si no hay información sobre la pregunta en el contexto, responde sólo 'No lo sé'.
        """),
        ("human", "Contexto: {context}.\nPregunta del empleado: {input}")
    ]
)

document_chain = create_stuff_documents_chain(llm, prompt_rag)

ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
def busqueda_de_respuestas_RAG(pregunta) -> Dict:
  documentos_relacionados = retriever.invoke(pregunta)

  if not documentos_relacionados:
    return {
        "respuesta": "No lo sé.",
        "citaciones": [],
        "documentos_encontrados": False
        }

  answer = document_chain.invoke({
      "input": pregunta,
      "context": documentos_relacionados
  })

  if answer.rstrip(".!?") == 'No lo sé':
    return {
        "respuesta": "No lo sé.",
        "citaciones": [],
        "documentos_encontrados": False
        }

  return {
        "respuesta": answer,
        "citaciones": documentos_relacionados,
        "documentos_encontrados": True
        }




In [ ]:
r = busqueda_de_respuestas_RAG("¿Puedo obtener un reembolso por el internet de mi home office?")
print(r)

In [ ]:
len(r["citaciones"])

In [ ]:
mensajes_de_prueba = [
	"¿Puedo obtener un reembolso por el internet de mi home office?",
	"Quiero una excepción para teletrabajar durante 5 días.",
	"¿Cómo funciona la política de comidas para viajes?",
	"¿Existe una política para anticipos de vacaciones?",
	"¿Quién fue Napoleón Bonaparte?"
]

In [ ]:
for pregunta in mensajes_de_prueba:
  respuesta_RAG = busqueda_de_respuestas_RAG(pregunta)
  print(f"PREGUNTA: {pregunta}")
  print(f"RESPUESTA: {respuesta_RAG['respuesta']}")
  if respuesta_RAG['documentos_encontrados']:
    for i, citacion in enumerate(respuesta_RAG['citaciones']):
      print(f"    - CITACIÓN {i + 1}:")
      print(f"      Camino del documento: {citacion.metadata['file_path']}")
      print(f"      Contenido: {citacion.page_content.replace('\n', '')}")
  print("----------------------------------------------------------------")

# Agente con LangGraph

In [ ]:
!pip install -q langgraph

In [ ]:
from typing import TypedDict, Optional

class AgentState(TypedDict, total = False):
  pregunta: str
  triaje: dict
  respuesta: Optional[str]
  citaciones: Optional[list]
  rag_exito: bool
  accion_final: str

In [ ]:
def nodo_triaje(state: AgentState) -> AgentState:
  print("Ejecutando nodo 'triaje'...")
  return {"triaje": triaje(state["pregunta"])}

In [ ]:
def nodo_auto_resolver(state: AgentState) -> AgentState:
  print("Ejecutando nodo 'auto_resolver'...")
  respuesta_RAG = busqueda_de_respuestas_RAG(state["pregunta"])

  update: AgentState = {
      "respuesta": respuesta_RAG["respuesta"],
      "citaciones": respuesta_RAG["citaciones"],
      "rag_exito": respuesta_RAG["documentos_encontrados"]
  }

  if respuesta_RAG["documentos_encontrados"]:
    update["accion_final"] = "AUTO_RESOLVER"

  return update

In [ ]:
def nodo_pedir_info(state: AgentState) -> AgentState:
  print("Ejecutando nodo 'pedir_info'...")
  return {
      "respuesta": "Necesito más informaciones sobre tu pedido.",
      "citaciones": [],
      "accion_final": "PEDIR_INFO"
  }

In [ ]:
def nodo_abrir_ticket(state: AgentState) -> AgentState:
  print("Ejecutando nodo 'abrir_ticket'...")

  tri = state["triaje"]

  return {
      "respuesta": f"Abrir ticket con urgencia {tri['urgencia']}. Pedido: {state['pregunta']}.",
      "citaciones": [],
      "accion_final": "ABRIR_TICKET"
  }

In [ ]:
def arista_decision_triaje(state: AgentState) -> str:
  print("Decidiendo el flujo después del nodo 'triaje'...")
  tri = state["triaje"]

  if tri["decision"] == "AUTO_RESOLVER":
    return "rag"
  elif tri["decision"] == "PEDIR_INFO":
    return "info"
  else:
    return "ticket"

In [ ]:
def arista_decision_rag(state: AgentState) -> str:
  print("Decidiendo el flujo después del nodo 'auto_resolver'...")

  if state["rag_exito"]:
    print("RAG con éxito, finalizando el flujo.")
    return "ok"

  KEYWORDS_ABRIR_TICKET = ["aprobación", "aprobar", "excepción", "liberación", "autorización",
                         "autorizar", "abrir ticket", "acceso especial"]

  if any(keyword in state["pregunta"].lower() for keyword in KEYWORDS_ABRIR_TICKET):
    print("RAG ha fallado, pero hay palabras relacionadas con abrir ticket.")
    return "ticket"

  print("RAG ha fallado, pediré más informaciones al usuario.")
  return "info"

In [ ]:
from langgraph.graph import START, END, StateGraph

workflow = StateGraph(AgentState)

workflow.add_node("triaje", nodo_triaje)
workflow.add_node("auto_resolver", nodo_auto_resolver)
workflow.add_node("pedir_info", nodo_pedir_info)
workflow.add_node("abrir_ticket", nodo_abrir_ticket)

workflow.add_edge(START, "triaje")
workflow.add_conditional_edges("triaje", arista_decision_triaje, {
    "rag": "auto_resolver",
    "info": "pedir_info",
    "ticket": "abrir_ticket"
})

workflow.add_conditional_edges("auto_resolver", arista_decision_rag, {
    "info": "pedir_info",
    "ticket": "abrir_ticket",
    "ok": END
})

workflow.add_edge("pedir_info", END)
workflow.add_edge("abrir_ticket", END)

grafo = workflow.compile()

In [ ]:
from IPython.display import display, Image

graph_bytes = grafo.get_graph().draw_mermaid_png()
display(Image(graph_bytes))

In [ ]:
PREGUNTA = "Puedo reembolsar mi internet?"

respuesta = grafo.invoke({"pregunta": PREGUNTA})
print("")
print(f"PREGUNTA: {PREGUNTA}")
print(f"DECISIÓN DE TRIAJE: {respuesta['triaje']['decision']} | URGENCIA: {respuesta['triaje']['urgencia']} | ACCIÓN FINAL: {respuesta['accion_final']}")
print(f"RESPUESTA: {respuesta['respuesta']}")
if respuesta['citaciones']:
  for i, citacion in enumerate(respuesta['citaciones']):
    print(f"    - CITACIÓN {i + 1}:")
    print(f"      Camino del documento: {citacion.metadata['file_path']}")
    print(f"      Contenido: {citacion.page_content.replace('\n', '')}")

In [ ]:
mensajes_de_prueba = [
	"¿Puedo obtener un reembolso por el internet de mi home office?",
	"Quiero una excepción para teletrabajar durante 5 días.",
	"¿Cómo funciona la política de comidas para viajes?",
	"¿Existe una política para anticipos de vacaciones?",
	"¿Quién fue Napoleón Bonaparte?"
]

In [ ]:
for prueba in mensajes_de_prueba:
  respuesta = grafo.invoke({"pregunta": prueba})
  print("")
  print(f"PREGUNTA: {prueba}")
  print(f"DECISIÓN DE TRIAJE: {respuesta['triaje']['decision']} | URGENCIA: {respuesta['triaje']['urgencia']} | ACCIÓN FINAL: {respuesta['accion_final']}")
  print(f"RESPUESTA: {respuesta['respuesta']}")
  if respuesta['citaciones']:
    for i, citacion in enumerate(respuesta['citaciones']):
      print(f"    - CITACIÓN {i + 1}:")
      print(f"      Camino del documento: {citacion.metadata['file_path']}")
      print(f"      Contenido: {citacion.page_content.replace('\n', '')}")
  print("-----------------------------------------------")